# Mean molecular weight and electron to proton ratios from abundance tables

To try and calculate the electron to proton (or electron to hydrogen really) ratio for the different abundance tables available in XSPEC, as well as the mean molecular weight. Pretty commonly used in X-ray galaxy cluster analyses, but wasn't actually that sure how it was calculated. I'm pretty sure this is correct, but I am not guaranteeing it. 

The abundance table file is found in the "spectral/manager" path in the HEASoft directory.

In [7]:
import pandas as pd
import numpy as np
import json

## Loading the abundance file from XSPEC (from Heasoft 6.36)

In [8]:
abund_tabs = pd.read_csv('abundances.dat', 
                         sep=r"\s+",
                         skiprows=0,  
                         nrows=10,        # 10 abundance rows (feld through felc) before the References section
                         index_col=0,     # use the row labels (feld:, angr:, ...) as the index
                         header=0, ).T
abund_tabs.columns = abund_tabs.columns.str.rstrip(":")
abund_tabs

elts:,feld,angr,aneb,grsa,wilm,lodd,aspl,lpgp,lpgs,felc
H,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000
He,9.770000e-02,9.770000e-02,8.010000e-02,8.510000e-02,9.770000e-02,7.920000e-02,8.510000e-02,8.410000e-02,9.690000e-02,0.079400
Li,1.260000e-11,1.450000e-11,2.190000e-09,1.260000e-11,0.000000e+00,1.900000e-09,1.120000e-11,1.260000e-11,2.150000e-09,0.000000
Be,2.510000e-11,1.410000e-11,2.870000e-11,2.510000e-11,0.000000e+00,2.570000e-11,2.400000e-11,2.400000e-11,2.360000e-11,0.000000
B,3.550000e-10,3.980000e-10,8.820000e-10,3.550000e-10,0.000000e+00,6.030000e-10,5.010000e-10,5.010000e-10,7.260000e-10,0.000000
C,3.980000e-04,3.630000e-04,4.450000e-04,3.310000e-04,2.400000e-04,2.450000e-04,2.690000e-04,2.450000e-04,2.780000e-04,0.000389
N,1.000000e-04,1.120000e-04,9.120000e-05,8.320000e-05,7.590000e-05,6.760000e-05,6.760000e-05,7.240000e-05,8.190000e-05,0.000100
O,8.510000e-04,8.510000e-04,7.390000e-04,6.760000e-04,4.900000e-04,4.900000e-04,4.900000e-04,5.370000e-04,6.060000e-04,0.000776
F,3.630000e-08,3.630000e-08,3.100000e-08,3.630000e-08,0.000000e+00,2.880000e-08,3.630000e-08,3.630000e-08,3.100000e-08,0.000000
Ne,1.290000e-04,1.230000e-04,1.380000e-04,1.200000e-04,8.710000e-05,7.410000e-05,8.510000e-05,1.120000e-04,1.270000e-04,0.000120


## Defining the number of electrons for each element

In [9]:
electrons = {en: en_ind+1 for en_ind, en in enumerate(abund_tabs.index)}
elec_arr = np.array(list(electrons.values()))
electrons

{'H': 1,
 'He': 2,
 'Li': 3,
 'Be': 4,
 'B': 5,
 'C': 6,
 'N': 7,
 'O': 8,
 'F': 9,
 'Ne': 10,
 'Na': 11,
 'Mg': 12,
 'Al': 13,
 'Si': 14,
 'P': 15,
 'S': 16,
 'Cl': 17,
 'Ar': 18,
 'K': 19,
 'Ca': 20,
 'Sc': 21,
 'Ti': 22,
 'V': 23,
 'Cr': 24,
 'Mn': 25,
 'Fe': 26,
 'Co': 27,
 'Ni': 28,
 'Cu': 29,
 'Zn': 30}

## Calculating the electron-to-proton (or electron-to-hydrogen) ratio

In [10]:
def calc_e_to_p_ratio(abund_table, result_dict, assumed_abund=0.3):
    ep_hy = 1
    ep_he = abund_tabs[abund_table]['He']*elec_arr[1]
    ep_met = ((abund_tabs[abund_table]['Li':].values / (1+abund_tabs[abund_table]['He'])) * assumed_abund * elec_arr[2:]).sum()

    elec_to_prot = float(ep_hy + ep_he + ep_met)
    
    
    result_dict[abund_table] = round(elec_to_prot, 3)
    
    print(f'e-to-p = {elec_to_prot:.3f}' +  '\n')

## Mean molecular weight - **assuming fully ionized plasma**
(2X + 0.75Y + 0.56Z)$^{-1}$ is the expression for the approximate mean molecular weight of a fully-ionized plasma. 

X is the mass fraction of hydrogen, Y is the mass fraction of helium, and Z is the mass fraction of eveeeerything else.

That ^ expression involves an average of number of protons+neutrons for metals (value is 15.5)

In [11]:
def calc_mean_mol_weight(abund_table, result_dict, assumed_met=0.3):
    # We are multipling the number of electrons by two to get the total neutron+protons
    he_contrib = abund_tabs[abund_table]['He']*2*electrons['He']

    # Contribution of everything but H and He (i.e. the metals) - note that we renormalise the abundances to be in terms of 
    #  the H + He abundances, and THEN apply the assumed metallicity (because the metallicity definition is not just relative to Hydrogen
    #  like these abundance tables are)
    met_contrib = ((abund_tabs[abund_table]['Li':].values / (1+abund_tabs[abund_table]['He']))*assumed_met * 2*elec_arr[2:]).sum()

    hy_mass_frac = 1/(1 + he_contrib + met_contrib)
    he_mass_frac = he_contrib/(1 + he_contrib + met_contrib)
    met_mass_frac = met_contrib/(1 + he_contrib + met_contrib)
    print('MASS FRACTIONS\n', 'X={}\n'.format(hy_mass_frac), 'Y={}\n'.format(he_mass_frac), 'Z={}\n'.format(met_mass_frac), sep='')

    # Finally, the actual number that we want, the mean molecular weight - calculated assuming a fully ionized plasma, don't 
    #  include the derivation of this here though
    mean_mol_weight = 1/((2*hy_mass_frac) + (0.75*he_mass_frac) + (0.56*met_mass_frac))
    
    result_dict[abund_table] = round(mean_mol_weight, 3)

    print('MEAN MOLECULAR WEIGHT\n', "μ={}".format(mean_mol_weight.round(3)), sep='')

## Saving the results

In [12]:
# Making dictionaries to append the results to, that can be read in by XGA
mean_mol_weights = {}
e_to_p_ratios = {}

### Anders and Grevasse (angr)

In [13]:
calc_e_to_p_ratio('angr', e_to_p_ratios)
calc_mean_mol_weight('angr', mean_mol_weights)

e-to-p = 1.199

MASS FRACTIONS
X=0.7151634161021603
Y=0.2794858630127242
Z=0.005350720885115523

MEAN MOLECULAR WEIGHT
μ=0.609


### Asplund et al. (aspl)

In [14]:
calc_e_to_p_ratio('aspl', e_to_p_ratios)
calc_mean_mol_weight('aspl', mean_mol_weights)

e-to-p = 1.173

MASS FRACTIONS
X=0.7432715051268192
Y=0.2530096203451692
Z=0.003718874528011619

MEAN MOLECULAR WEIGHT
μ=0.596


### Feldman et al. 1992 (feld)

In [15]:
calc_e_to_p_ratio('feld', e_to_p_ratios)
calc_mean_mol_weight('feld', mean_mol_weights)

e-to-p = 1.199

MASS FRACTIONS
X=0.7152118157440713
Y=0.279504777592783
Z=0.005283406663145672

MEAN MOLECULAR WEIGHT
μ=0.609


### Anders E. and Ebihara 1982 (aneb)

In [16]:
calc_e_to_p_ratio('aneb', e_to_p_ratios)
calc_mean_mol_weight('aneb', mean_mol_weights)

e-to-p = 1.164

MASS FRACTIONS
X=0.753220518757761
Y=0.24133185420998662
Z=0.005447627032252514

MEAN MOLECULAR WEIGHT
μ=0.592


### Grevesse and Sauval (grsa)

In [17]:
calc_e_to_p_ratio('grsa', e_to_p_ratios)
calc_mean_mol_weight('grsa', mean_mol_weights)

e-to-p = 1.173

MASS FRACTIONS
X=0.7425135500950001
Y=0.252751612452338
Z=0.004734837452661864

MEAN MOLECULAR WEIGHT
μ=0.596


### Wilms et al. 2000 (wilm)

In [18]:
calc_e_to_p_ratio('wilm', e_to_p_ratios)
calc_mean_mol_weight('wilm', mean_mol_weights)

e-to-p = 1.198

MASS FRACTIONS
X=0.7166458645675048
Y=0.28006520387298084
Z=0.003288931559514407

MEAN MOLECULAR WEIGHT
μ=0.608


### Lodders et al. 2003; photospheric (lodd)

In [19]:
calc_e_to_p_ratio('lodd', e_to_p_ratios)
calc_mean_mol_weight('lodd', mean_mol_weights)

e-to-p = 1.161

MASS FRACTIONS
X=0.7566122017664483
Y=0.23969474551961084
Z=0.0036930527139409173

MEAN MOLECULAR WEIGHT
μ=0.59


### Lodders et al. 2009; photospheric (lpgp)

In [20]:
calc_e_to_p_ratio('lpgp', e_to_p_ratios)
calc_mean_mol_weight('lpgp', mean_mol_weights)

e-to-p = 1.171

MASS FRACTIONS
X=0.7453560416780702
Y=0.2507377724205028
Z=0.0039061859014269044

MEAN MOLECULAR WEIGHT
μ=0.595


### Lodders et al. 2009; proto-solar (lpgs)

In [21]:
calc_e_to_p_ratio('lpgs', e_to_p_ratios)
calc_mean_mol_weight('lpgs', mean_mol_weights)

e-to-p = 1.197

MASS FRACTIONS
X=0.7176252992591481
Y=0.2781515659928458
Z=0.004223134748006072

MEAN MOLECULAR WEIGHT
μ=0.607


In [22]:
e_to_p_ratios

{'angr': 1.199,
 'aspl': 1.173,
 'feld': 1.199,
 'aneb': 1.164,
 'grsa': 1.173,
 'wilm': 1.198,
 'lodd': 1.161,
 'lpgp': 1.171,
 'lpgs': 1.197}

In [23]:
with open("../../xga/files/e_to_p_ratios.json", "w") as f:
    json.dump(e_to_p_ratios, f)

In [24]:
mean_mol_weights

{'angr': np.float64(0.609),
 'aspl': np.float64(0.596),
 'feld': np.float64(0.609),
 'aneb': np.float64(0.592),
 'grsa': np.float64(0.596),
 'wilm': np.float64(0.608),
 'lodd': np.float64(0.59),
 'lpgp': np.float64(0.595),
 'lpgs': np.float64(0.607)}

In [25]:
with open("../../xga/files/mean_mol_weights.json", "w") as f:
    json.dump(mean_mol_weights, f)